# 面波模型、波场、记录与频散对比重绘

默认读取六组20Hz面波实验，可修改算例列表和输入路径。


当前显示：道集gray、模型turbo、波场coolwarm。15/30 Hz默认9时刻全域Vz，统一±20 µm/s对称对数色标；原始DEM保留jet，频散谱独立使用viridis。参数见同目录styles/current_publication.json。


## 输入与输出约定

这里的代码只读输入，不重跑模拟，不修改原始SEG-Y/VTS。路径可以是绝对路径；相对路径以项目外层工作区为基准。生成的图件、CSV、缓存及执行副本统一放入外层`out`的子目录。

如果输入不是默认文件夹结构，可在配置中为每个算例单独填写`record_files`、`geometry_csv`、`model_vts`、`snapshot_index_csv`和`snapshot_dir`，详见同目录README。源码中的Notebook不保存运行输出；需要保留执行结果时请另存到`out`。


In [ ]:
import sys, json
from pathlib import Path
sys.dont_write_bytecode = True

# 如果Notebook从其他位置启动，可在这里指定仓库根目录。
REPOSITORY_ROOT = None
if REPOSITORY_ROOT is None:
    candidates = [p for q in [Path.cwd(), *Path.cwd().parents]
                  for p in [q, q / "SeisWaveLab_Curvilinear"]
                  if (p / "tools/seismic_plotting/seisplot/core.py").is_file()]
    if not candidates:
        raise FileNotFoundError("请将REPOSITORY_ROOT设为Git仓库根目录")
    REPOSITORY_ROOT = candidates[0].resolve()
else:
    REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()
WORKSPACE = REPOSITORY_ROOT.parent
TOOL_DIR = REPOSITORY_ROOT / "tools/seismic_plotting"
sys.path.insert(0, str(TOOL_DIR))
from seisplot.core import load_config, gallery
SHOW_INLINE = True


In [ ]:
from seisplot.surface_wave import SurfaceWaveStudy
CONFIG = load_config(TOOL_DIR / "examples/surface_wave.json")
INPUT_RUN_DIR = WORKSPACE / "out/surface_wave_masw/runs/run_20261023_02"
THEORY_CURVE_FILE = WORKSPACE / "out/surface_wave_masw/theoretical_dispersion.csv"  # 设为None可不叠加理论
OUTPUT_DIR = WORKSPACE / "out/replots/surface_wave"
CONFIG["datasets"][0]["root"] = str(INPUT_RUN_DIR)
CONFIG["theory_curve_csv"] = str(THEORY_CURVE_FILE) if THEORY_CURVE_FILE else None
CONFIG["output_dir"] = str(OUTPUT_DIR)
CONFIG["snapshots"]["times_s"] = [0.2, 0.6, 1.0, 1.5, 2.0]
CONFIG["snapshots"]["crop_m"] = [-20, 280, 0, 100]  # 相对震源x及物理深度，单位m
RUN_MODELS = True
RUN_WAVEFIELDS = True

# 显示样式由styles/current_publication.json统一维护；修改CONFIG可显式覆盖。
print("显示配置:", CONFIG["style_profile"], CONFIG["style_provenance"]["version"])


## 1. 检查记录和接收几何


In [ ]:
study = SurfaceWaveStudy(CONFIG)
print(json.dumps(study.inspect(), ensure_ascii=False, indent=2))


## 2. 原始与300ms AGC记录


In [ ]:
study.plot_records()


## 3. 速度模型和裁切波场

默认相对震源x=-20–280m、深度0–100m，空间1:1。裁切不修改模拟域。同一目标时刻共用SymLog范围，不同目标时刻分别定标。


In [ ]:
if RUN_MODELS:
    study.plot_models()
if RUN_WAVEFIELDS:
    study.plot_wavefields()


## 4. 边界、网格残差与原始道对比

必须使用匹配的时间采样和偏移距。加深、加宽是相对有限参考域的差异，不能直接称为无边界真解误差。


In [ ]:
study.plot_record_differences()
study.plot_trace_overlay()


## 5. 重新计算频散谱

从原始记录作相位归一化与相移叠加。代码不使用理论速度门限，导出的是全速度范围峰值，仍需独立确认模态。频率取样间隔不是实际分辨率，后者受记录时长和孔径限制。


In [ ]:
study.compute_dispersion()
study.plot_dispersion()
study.plot_offset_windows()
study.plot_phase_curves()


## 6. 导出与预览

`data/*unguided_picks.csv`可用作曲线检查与反演输入准备。理论CSV只用于叠加显示。


In [ ]:
print("输出目录：", study.out)
if SHOW_INLINE:
    gallery(study.out, "dispersion*.png", limit=2)
